In [50]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import yaml, os, copy, time, json, re

from scipy.stats import poisson, qmc
from scipy.optimize import root_scalar, brentq
from scipy.special import ellipe, ellipk
from mpmath import polylog
from math import comb
from pathlib import Path

from gpt import GPT

from distgen import Generator
from matplotlib.colors import SymLogNorm

from specific_particle_tracer import SpecificParticleTracer, HemisphericalTip, FlatCathode
from specific_particle_tracer.bem.geometry import HemisphericalTipBEMGeometry, CylindricalWellBEMGeometry
from specific_particle_tracer.distributions import flat_distribution_to_hemisphere
from specific_particle_tracer.constants import ELEMENTARY_CHARGE

from specific_particle_tracer.plotting import plot_profiles, static_potential_grid, image_potential_grid

from GPT_tools.tools import get_screen_data
from GPT_tools.gpt_plot import gpt_plot_dist1d, gpt_plot_dist2d
from GPT_tools.image_charge import MakeMetalParticleGroup, MakeSemiconductorParticleGroup, MakeEnergyOffsetParticleGroup
from GPT_tools.image_charge import MTE_model, QE_model, getValueFromSettings, getSemiconductorEexc

template_dir = '/nfs/bbl/online/coulomb/template/'
DISTGEN_INPUT_FILE = os.path.join(template_dir,'distgen.in.yaml')

# ---------------------------------------------------------------------
# Settings for multi-threading

max_workers = 90   # number of threads on your computer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
# Some font size defaults
plt.rcParams.update({
    'font.size': 14,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'mathtext.fontset': 'dejavusans',
    'axes.labelsize': 18,
    'axes.titlesize': 18,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

In [44]:
# Functions to load data

def load_data(file_to_load, save_dir='/nfs/bbl/online/coulomb/saves/', load_gpt_n=True, verbose=True, default_old_n_runs=1000):
 
    dir_name = os.path.join(save_dir, file_to_load)    
    
    M = np.loadtxt(os.path.join(dir_name, 'M.txt'))
    max_particles = M.shape[0] - 1
    
    with open(os.path.join(dir_name, 'settings.json'), "r") as f:
        settings = json.load(f)

    settings['max_particles'] = max_particles

    #M[np.triu_indices(M.shape[0], k=1)] = 0 # kill unphysical upper triangular part of matrix, just in case

    if ('counts_n' in settings):
        counts_n = np.array(settings['counts_n']).astype(int)

        if (counts_n.shape != M.shape):
            raise RuntimeError('counts_n has wrong shape')

        del settings['counts_n']
    else:
        counts_n = np.zeros(M.shape, dtype=int)
        counts_n[0,0] = 1

        for ii in np.arange(1, max_particles+1):

            if (np.any(M[ii,:] != 0)):
                counts_n[ii,:] = np.round(default_old_n_runs*M[ii,:]).astype(int)

                diff = default_old_n_runs - np.sum(counts_n[ii,:])
                if (diff != 0):
                    counts_n[ii,np.argmax(M[ii,:])] += diff

        #settings['counts_n'] = counts_n

    if ('n_runs_n' in settings):
        n_runs_n = np.array(settings['n_runs_n']).astype(int)

        if (len(n_runs_n) != M.shape[0]):
            raise RuntimeError('n_runs_n has wrong length')

        del settings['n_runs_n']

    else:
        n_runs_n = np.zeros(max_particles+1, dtype=int)

        for ii in np.arange(1, max_particles+1):
            if (np.any(M[ii,:] != 0)):
                n_runs_n[ii] = np.sum(counts_n[ii,:])

        #settings['n_runs_n'] = n_runs_n

    if ('rel_accuracy' not in settings):
        settings['rel_accuracy'] = None

    if ('n_runs_start' not in settings):
        settings['n_runs_start'] = default_old_n_runs

    if ('n_runs_min_add' not in settings):
        settings['n_runs_min_add'] = default_old_n_runs

    if ('n_runs_max_times_more' not in settings):
        settings['n_runs_max_times_more'] = None

    if ('only_survivors' not in settings):
        if (verbose):
            print("Please manually set the value of settings['only_survivors']")
    
    def gpt_index(path):
        m = re.search(r"gpt_data_(\d+)\.h5$", path.name)
        return int(m.group(1)) if m else float("inf")
    
    files_to_load = sorted(Path(dir_name).glob("gpt_data_*.h5"), key=gpt_index)
        
    gpt_data_n = [None]*(max_particles+1)
    
    if (load_gpt_n):
    
        for gpt_file_to_load in files_to_load:
    
            gpt_file_to_load = Path(gpt_file_to_load)
            match = re.search(r'gpt_data_(\d+)\.h5$', gpt_file_to_load.name)
    
            if (match is None):
                print(f'Skipping: {gpt_file_to_load}')
                continue
    
            num_par = int(match.group(1))

            if (num_par > max_particles):
                print(f'Skipping: {gpt_file_to_load}')
                continue
    
            if (gpt_data_n[num_par] is not None):
                raise RuntimeError(f'gpt_data_n[{num_par}] already loaded')

            if (verbose):
                print(f'Loading from: {gpt_file_to_load}')
            gpt_data_n[num_par] = GPT.from_archive(str(gpt_file_to_load))
    
        missing_gpt_data = [ii for ii,g in enumerate(gpt_data_n) if (ii != 0 and g is None)]
    
        if (len(missing_gpt_data) > 0):
            print(f'Missing GPT data for: {missing_gpt_data}')

    if (verbose):
        for k,v in settings.items():
            if (k == 'counts_n'):
                print(f"{str(k):25s} array with shape {counts_n.shape}")
            elif (k == 'n_runs_n'):
                print(f"{str(k):25s} {n_runs_n}")
            else:
                print(f"{str(k):25s} {v}")
    
    return (counts_n, settings, gpt_data_n)


def check_loaded_rows(M, counts_n, n_runs_n, gpt_data_n, settings, check_gpt=True, reset_bad=True):

    bad_rows = []
    repaired_rows = []

    M[0,:] = 0
    M[0,0] = 1.0
    counts_n[0,:] = 0
    counts_n[0,0] = 1

    for num_par in np.arange(1, M.shape[0]):

        N_counts = int(np.sum(counts_n[num_par,:]))
        N_stored = int(n_runs_n[num_par])

        if (N_counts == 0 and N_stored == 0):
            continue

        row_bad = False

        if (N_counts != N_stored):
            print(f'Problem in row {num_par}: counts_n sums to {N_counts}, n_runs_n is {N_stored}')
            row_bad = True

        else:
            M_from_counts = counts_n[num_par,:] / N_counts

            if (not np.allclose(M[num_par,:], M_from_counts, rtol=1e-12, atol=1e-12)):
                M[num_par,:] = M_from_counts
                repaired_rows.append(num_par)

        if ((not row_bad) and check_gpt):

            g = gpt_data_n[num_par]

            if (g is None):

                if (counts_n[num_par,0] != N_stored):
                    print(f'Problem in row {num_par}: gpt_data_n is None, but nonzero survivors exist')
                    row_bad = True

            else:

                try:
                    counts_from_gpt = get_counts_from_gpt_data(g, num_par, N_stored, settings)
                except Exception as err:
                    print(f'Problem in row {num_par}: could not reconstruct counts from gpt_data_n')
                    print(err)
                    row_bad = True

                if (not row_bad):
                    if (not np.array_equal(counts_from_gpt.astype(int), counts_n[num_par,:].astype(int))):
                        print(f'Problem in row {num_par}: gpt_data_n counts do not match counts_n')
                        row_bad = True

        if (row_bad):
            bad_rows.append(num_par)

            if (reset_bad):
                reset_row(num_par, M, counts_n, n_runs_n, gpt_data_n)

    if (len(repaired_rows) > 0):
        print(f'Recomputed M from counts_n for rows: {repaired_rows}')

    if (len(bad_rows) > 0):
        if (reset_bad):
            print(f'Reset bad rows: {bad_rows}')
        else:
            print(f'Bad rows found: {bad_rows}')
    else:
        print('All loaded rows look self-consistent')

    return bad_rows, repaired_rows

def reset_row(num_par, M, counts_n, n_runs_n, gpt_data_n):

    num_par = int(num_par)

    M[num_par,:] = 0
    counts_n[num_par,:] = 0
    n_runs_n[num_par] = 0
    gpt_data_n[num_par] = None

def find_lambda_for_avg_final(M, desired_avg_final, lambda_max=None, n_sigma=2.0):

    def avg_final_electrons_from_lambda(M, lam):

        initial_n = np.arange(M.shape[0])
        final_m = np.arange(M.shape[1])

        avg_final_given_n = M @ final_m
        p_initial = poisson.pmf(initial_n, lam)

        return np.sum(p_initial * avg_final_given_n)
    
    if (desired_avg_final < 0):
        raise ValueError("desired_avg_final must be nonnegative.")

    if (desired_avg_final == 0):
        return 0.0

    if (lambda_max is None):
        max_initial_n = M.shape[0] - 1
        lambda_max = 0.5 * (
            n_sigma**2
            + 2 * max_initial_n
            - n_sigma * np.sqrt(n_sigma**2 + 4 * max_initial_n)
        )

    def f(lam):
        return avg_final_electrons_from_lambda(M, lam) - desired_avg_final

    f0 = f(0.0)
    fmax = f(lambda_max)

    if (f0 > 0):
        raise ValueError("The desired average final electron number is already exceeded at lambda=0.")

    if (fmax < 0):
        raise ValueError(
            "Could not reach the desired average final electron number within "
            f"lambda_max={lambda_max:.6g}. Try increasing lambda_max, or check "
            "whether M has enough rows to represent larger initial electron counts."
        )

    lam_solution = brentq(f, 0.0, lambda_max)

    return lam_solution



def get_gpt_data_at_n(avg_n_final, C, gpt_data_n, poisson_factor_cutoff=1e-2):
    # ---------------------------------------------------------------------
    # Make gpt_data object for a given injected fluence, with poisson statistics
    # This is intended to allow you to, say, make a screen image with a given laser fluence
    # Note: particle "charges" will now not be just electron charge, but weighted by poisson factors
    
    # ---------------------------------------------------------------------

    M = M_from_C(C)
    n_runs_n=counts_from_C(C)
    
    avg_n_injected = find_lambda_for_avg_final(M, avg_n_final)
    print(f'Using avg_n_injected = {avg_n_injected}')

    assert len(gpt_data_n) == M.shape[0]
    assert gpt_data_n[0] is None
    assert gpt_data_n[1] is not None

    if (n_runs_n is None):
        n_runs_n = np.zeros(M.shape[0], dtype=int)
        n_runs_n[1:] = 1000
    else:
        n_runs_n = np.array(n_runs_n).astype(int)

    assert len(n_runs_n) == M.shape[0]

    if (n_runs_n[1] == 0):
        raise RuntimeError('n_runs_n[1] is zero')
    
    # ---------------------------------------------------------------------
    
    lam = avg_n_injected
    used_settings = gpt_data_n[1].input['variables']
    ztol = used_settings['zmax'] / np.max([1,used_settings['n_screens']]) / 100  

    omitted_tail = poisson.sf(len(gpt_data_n)-1, lam)
    if (omitted_tail > poisson_factor_cutoff):
        print(f'Warning: omitted Poisson tail probability = {omitted_tail}')
    
    gpt_data = copy.deepcopy(gpt_data_n[1])

    pois_factor = poisson.pmf(1, lam)

    for jj,p in enumerate(gpt_data.particles):
        p.weight = p.weight * pois_factor / n_runs_n[1]

    id_offsets = np.zeros(len(gpt_data_n), dtype=int)
    next_id_offset = 0

    for num_par in np.arange(1, len(gpt_data_n)):

        g = gpt_data_n[num_par]

        if (g is None):
            id_offsets[num_par] = next_id_offset
            continue

        id_offsets[num_par] = next_id_offset

        max_id = 0
        for p in g.particles:
            if (len(p.id) > 0):
                max_id = np.max([max_id, np.max(p.id)])

        next_id_offset = next_id_offset + int(max_id)
    
    for num_par in np.arange(2, len(gpt_data_n)):

        pois_factor = poisson.pmf(num_par, lam)
        
        if (pois_factor > poisson_factor_cutoff):

            g = gpt_data_n[num_par]

            if (g is None):
                raise RuntimeError(f'gpt_data_n[{num_par}] is None')

            if (n_runs_n[num_par] == 0):
                raise RuntimeError(f'n_runs_n[{num_par}] is zero')
    
            for jj,p in enumerate(gpt_data.particles):

                z = p['mean_z']
                p_new = get_screen_data(g, screen_z=z)[0]

                if (np.abs(p_new['mean_z'] - z) < ztol):
                    p_new.weight = p_new.weight * pois_factor / n_runs_n[num_par]
                    p_new.id = p_new.id + id_offsets[num_par]
                    gpt_data.particles[jj] = gpt_data.particles[jj] + p_new   

    return gpt_data

In [3]:
# Some plotting functions

def collect_trajectories(trajectories, initial):
    """Return {id: (t_array, pos_array (n,3), mom_array (n,3))} by following each
    particle's id across the list of per-time-snapshot ParticleGroups, starting
    from its initial state in `initial`."""
    by_id = {}
    for i, pid in enumerate(initial.id):
        by_id[pid] = (
            [initial.t[i]],
            [(initial.x[i], initial.y[i], initial.z[i])],
            [(initial.px[i], initial.py[i], initial.pz[i])],
        )
    for traj in trajectories:
        for i, pid in enumerate(traj.id):
            t_list, pos_list, mom_list = by_id.setdefault(pid, ([], [], []))
            if t_list and traj.t[i] == t_list[0]:
                continue  # snapshot landed exactly on the birth time already prepended
            t_list.append(traj.t[i])
            pos_list.append((traj.x[i], traj.y[i], traj.z[i]))
            mom_list.append((traj.px[i], traj.py[i], traj.pz[i]))
    return {
        pid: (np.array(t_list), np.array(pos_list), np.array(mom_list))
        for pid, (t_list, pos_list, mom_list) in by_id.items()
    }

def M_from_C(C):
    return C / np.sum(C, axis=1, keepdims=True)

def counts_from_C(C):
    return np.sum(C, axis=1)

def truncate_C(C):
    row_is_done = np.any(C != 0, axis=1)
    rows_done = np.where(row_is_done)[0]
    if (len(rows_done) == 0):
        return C[0:0, 0:0]
    N = np.max(rows_done) + 1
    return C[:N, :N]
    
def get_fano_curve(M, lambda_list):

    initial_n = np.arange(M.shape[0])
    final_m = np.arange(M.shape[1])

    Bi = M @ final_m
    Ci = M @ (final_m**2)

    Pi = poisson.pmf(initial_n[:,None], lambda_list[None,:])

    avg_n = Bi @ Pi
    avg_n2 = Ci @ Pi

    fano = (avg_n2 - avg_n**2)/avg_n

    return avg_n, fano


def sample_M_dirichlet(C, rng=None):

    if (rng is None):
        rng = np.random.default_rng()

    M_sample = np.zeros_like(C, dtype=float)

    M_sample[0,0] = 1.0

    for ii in np.arange(1, C.shape[0]):

        allowed = np.arange(C.shape[1]) <= ii

        alpha = C[ii,allowed].astype(float)

        M_sample[ii,allowed] = rng.dirichlet(alpha)

    return M_sample

def get_avg_e_max_from_M(M, tail_prob=1e-3):

    max_initial_n = M.shape[0] - 1

    def f(lam):
        return poisson.sf(max_initial_n, lam) - tail_prob

    if (f(0) > 0):
        return 0.0

    lam_hi = max_initial_n

    while (f(lam_hi) < 0):
        lam_hi *= 2

    return brentq(f, 0, lam_hi)

def plot_fano_with_error(C, avg_e_max=None, use_electrons_escaping=True,
                         fig_ax=None, color='r', label='Data',
                         n_trials=50, ci=0.95, rng=None):

    C = truncate_C(C)
    M = M_from_C(C)
    
    if (rng is None):
        rng = np.random.default_rng()

    if (avg_e_max is None):
        avg_e_max = get_avg_e_max_from_M(M)

    lambda_list = np.linspace(0.01, avg_e_max, 300)

    charge_list, fano_list = get_fano_curve(M, lambda_list)

    if (use_electrons_escaping):
        x = charge_list
    else:
        x = lambda_list

    fano_mc = np.zeros((n_trials, len(lambda_list)))

    for kk in np.arange(n_trials):

        M_sample = sample_M_dirichlet(C, rng=rng)
        charge_sample, fano_sample = get_fano_curve(M_sample, lambda_list)

        if (use_electrons_escaping):
            order = np.argsort(charge_sample)
            x_sample = charge_sample[order]
            y_sample = fano_sample[order]

            keep = np.insert(np.diff(x_sample) > 0, 0, True)

            fano_mc[kk,:] = np.interp(x, x_sample[keep], y_sample[keep],
                                      left=np.nan, right=np.nan)
        else:
            fano_mc[kk,:] = fano_sample

    qlo = 50*(1 - ci)
    qhi = 100 - qlo

    fano_lo = np.nanpercentile(fano_mc, qlo, axis=0)
    fano_hi = np.nanpercentile(fano_mc, qhi, axis=0)

    if (fig_ax is None):
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    ax.fill_between(x, fano_lo, fano_hi, color=color, alpha=0.25, linewidth=0)
    ax.plot(x, fano_list, '-', label=label, color=color)

    ax.set_ylim((0,1))
    ax.set_xlim((0,np.nanmax(x)))

    if (use_electrons_escaping):
        ax.set_xlabel('Average electrons escaping')
    else:
        ax.set_xlabel('Average electrons emitted')

    ax.set_ylabel('Fano Factor')
    ax.legend()

def get_merit_curve(M, lambda_list):

    initial_n = np.arange(M.shape[0])

    p_one = np.zeros(len(lambda_list))
    p_multi = np.zeros(len(lambda_list))

    multi_given_n = np.sum(M[:,2:], axis=1)

    for jj,lam in enumerate(lambda_list):
        p_initial = poisson.pmf(initial_n, lam)

        p_one[jj] = np.sum(p_initial * M[:,1])
        p_multi[jj] = np.sum(p_initial * multi_given_n)

    return p_multi, p_one

def plot_merit(C, avg_e_max=None, fig_ax=None, color='r', label='Data',
               show_poisson=True, xlim=[0.01,1], n_trials=50, ci=0.95,
               rng=None):

    C = truncate_C(C)
    M = M_from_C(C)
    
    if (rng is None):
        rng = np.random.default_rng()

    if (avg_e_max is None):
        avg_e_max = get_avg_e_max_from_M(M)

    lambda_min = 1.0e-4
    lambda_list = np.geomspace(lambda_min, avg_e_max, 500)

    p_multi, p_one = get_merit_curve(M, lambda_list)

    mu_list = np.geomspace(1.0e-4, 20, 1000)

    p_one_ideal = mu_list * np.exp(-mu_list)
    p_multi_ideal = 1 - np.exp(-mu_list) * (1 + mu_list)

    if (fig_ax is None):
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    if (C is not None):

        p_one_mc = np.zeros((n_trials, len(lambda_list)))

        for kk in np.arange(n_trials):

            M_sample = sample_M_dirichlet(C, rng=rng)

            p_multi_sample, p_one_sample = get_merit_curve(M_sample, lambda_list)

            order = np.argsort(p_multi_sample)

            x_sample = p_multi_sample[order]
            y_sample = p_one_sample[order]

            keep = np.insert(np.diff(x_sample) > 0, 0, True)

            p_one_mc[kk,:] = np.interp(p_multi, x_sample[keep], y_sample[keep],
                                       left=np.nan, right=np.nan)

        qlo = 50*(1 - ci)
        qhi = 100 - qlo

        p_one_lo = np.nanpercentile(p_one_mc, qlo, axis=0)
        p_one_hi = np.nanpercentile(p_one_mc, qhi, axis=0)

        ax.fill_between(p_multi, p_one_lo, p_one_hi,
                        color=color, alpha=0.25, linewidth=0)

    ax.semilogx(
        p_multi,
        p_one,
        color=color,
        label=label
    )

    if (show_poisson):
        ax.semilogx(
            p_multi_ideal,
            p_one_ideal,
            color='k',
            linestyle='--',
            label=r'Poisson'
        )

    ax.set_xlim(xlim)
    ax.set_ylim((0,1))

    ax.set_ylabel(r'$P(N=1)$')
    ax.set_xlabel(r'$P(N>1)$')
    ax.legend()
    

In [4]:
# Run functions

def get_row_rel_err(counts_row):

    N = np.sum(counts_row)

    if (N == 0):
        return np.inf

    max_nonzero_count = np.max(counts_row[1:])

    if (max_nonzero_count == 0):
        return np.inf

    pmax = max_nonzero_count / N

    return np.sqrt((1 - pmax)/(N*pmax))


def get_next_n_runs(counts_row):

    N = int(np.sum(counts_row))

    if (N == 0):
        return n_runs_start

    if (N >= n_runs_max_total):
        return 0

    max_nonzero_count = np.max(counts_row[1:])

    if (max_nonzero_count == 0):
        n_more = np.max([n_runs_min_add, (n_runs_max_times_more - 1)*N])
        return int(np.min([n_more, n_runs_max_total - N]))

    pmax = max_nonzero_count / N
    n_required = int(np.ceil((1 - pmax)/(pmax*rel_accuracy**2)))
    n_more = n_required - N

    if (n_more <= 0):
        return 0

    n_more = np.max([n_runs_min_add, n_more])
    n_more = np.min([(n_runs_max_times_more - 1)*N, n_more])
    n_more = np.min([n_more, n_runs_max_total - N])

    return int(n_more)


def is_done(num_par):
    num_par = int(num_par)
    return ((n_runs_n[num_par] > 0) and
            ((get_row_rel_err(counts_n[num_par,:]) <= rel_accuracy) or
             (n_runs_n[num_par] >= n_runs_max_total)))


def get_values_to_scan():
    return [int(num_par) for num_par in values_to_scan_all if not is_done(num_par)]


def make_initial_particle_group(settings_run, rng=np.random.default_rng()):
    if (settings['cathode_type'] == 'metal'):
        PG_initial = MakeMetalParticleGroup(settings_run, DISTGEN_INPUT_FILE=DISTGEN_INPUT_FILE, verbose=False, only_survivors=settings_run['only_survivors'], rng=rng)
    elif (settings['cathode_type'] == 'semiconductor'):
        PG_initial = MakeSemiconductorParticleGroup(settings_run, DISTGEN_INPUT_FILE=DISTGEN_INPUT_FILE, verbose=False, only_survivors=settings_run['only_survivors'], rng=rng)
    elif (settings['cathode_type'] == 'distgen'):
        if (not settings_run['only_survivors']):
            raise ValueError("ERROR: only_survivors must be used with distgen")
        PG_initial = MakeEnergyOffsetParticleGroup(settings_run, DISTGEN_INPUT_FILE=DISTGEN_INPUT_FILE, verbose=False)
    else:
        raise ValueError("ERROR: bad cathode type")

    PG_initial.weight = ELEMENTARY_CHARGE # Just to make sure, not really needed
    if (settings_run['cathode_wall'] == 1):   
        PG_initial.z = -settings_run['H_cylinder']
    
    return PG_initial


def get_counts_from_gpt_data(gpt_data, num_par, n_runs_this_batch, settings_run):

    counts_this_batch = np.zeros(max_particles+1)

    if (len(gpt_data.screen) == 0):
        counts_this_batch[0] = n_runs_this_batch
        return counts_this_batch

    last_screen = get_screen_data(gpt_data, screen_z=settings_run['zmax'])[0]

    if (np.abs(last_screen['mean_z'] - settings_run['zmax']) > 1.0e-3*settings_run['zmax']):
        counts_this_batch[0] = n_runs_this_batch
        return counts_this_batch

    which_bunch = np.floor((last_screen.id-1)/num_par).astype(int)
    (vals, n_emit) = np.unique(which_bunch, return_counts=True)

    if (len(vals) == 0):
        counts_this_batch[0] = n_runs_this_batch
        return counts_this_batch

    if (np.min(which_bunch) < 0):
        raise RuntimeError('Bad particle IDs: found bunch index < 0')

    if (np.max(which_bunch) >= n_runs_this_batch):
        raise RuntimeError('Bad particle IDs: found bunch index >= n_runs_this_batch')

    if (np.max(n_emit) > num_par):
        raise RuntimeError('Bad particle IDs: found more survivors than injected particles')

    counts_this_batch[0] = n_runs_this_batch - len(vals)

    (emitted_vals, emitted_counts) = np.unique(n_emit, return_counts=True)
    counts_this_batch[emitted_vals] = emitted_counts

    if (np.sum(counts_this_batch) != n_runs_this_batch):
        raise RuntimeError('Bad emitted-count histogram')

    return counts_this_batch


def append_gpt_data(gpt_data_old, gpt_data_new, id_offset, settings_run):

    if (len(gpt_data_new.particles) == 0):
        return gpt_data_old

    gpt_data_new = copy.deepcopy(gpt_data_new)

    for p in gpt_data_new.particles:
        p.id = p.id + id_offset

    if (gpt_data_old is None):
        return gpt_data_new

    ztol = settings_run['zmax']*1.0e-6

    for p_new in gpt_data_new.particles:

        z_new = p_new['mean_z']
        found_screen = False

        for jj,p_old in enumerate(gpt_data_old.particles):

            if (np.abs(p_old['mean_z'] - z_new) < ztol):
                gpt_data_old.particles[jj] = p_old + p_new
                found_screen = True
                break

        if (not found_screen):
            gpt_data_old.particles.append(p_new)

    return gpt_data_old

## Distgen


In [34]:
settings = {}

# ---------------------------------------------------------------------
# BEM settings

settings['cathode_wall'] = 1

settings['field_max_length']=2.0e-9,
settings['image_max_length']=2.0e-9,
settings['rim_fillet_radius']=5.0e-9,
settings['bottom_fillet_radius']=1.5e-9

settings['R_cylinder'] = 5.0e-9 + settings['bottom_fillet_radius']
settings['H_cylinder'] = 10e-9

# ---------------------------------------------------------------------
# Settings for distgen

settings['random:type'] = 'pseudo'  # 'pseudo' = not Hammersley

settings['r_dist:sigma_xy:value'] = (settings['R_cylinder'] - settings['bottom_fillet_radius'])/2
settings['r_dist:sigma_xy:units'] = 'm'
settings['r_dist:alpha:value'] = 0  # 0 = flat top, 1 = gaussian

settings['t_dist:sigma_t:value'] = 10
settings['t_dist:sigma_t:units'] = 'fs'
settings['t_dist:alpha:value'] = 0  # 0 = flat top, 1 = gaussian

# ---------------------------------------------------------------------
# Settings for cathode emission model

settings['cathode_type'] = 'semiconductor'  # metal, semiconductor, or distgen
settings['only_survivors'] = False 

settings['effective_mass'] = 0.067 # 0.067

# Metal only settings
if (settings['cathode_type'] == 'metal'):
    settings['photon_energy:value'] = 1.600 # 1.461
    settings['photon_energy:units'] = 'eV'
    
    settings['kT:value'] = 25e-3
    settings['kT:units'] = 'meV'
    
    settings['work_function:value'] = 1.4
    settings['work_function:units'] = 'eV'

# Semiconductor only settings
if (settings['cathode_type'] == 'semiconductor'):
    settings['photon_energy:value'] = 1.500
    settings['photon_energy:units'] = 'eV'
    
    settings['electron_affinity:value'] = -0.0
    settings['electron_affinity:units'] = 'eV'
    
    settings['energy_gap:value'] = 1.4
    settings['energy_gap:units'] = 'eV'

# distgen only settings
if (settings['cathode_type'] == 'distgen'):
    settings['start:MTE:value'] = 25
    settings['start:MTE:units'] = 'meV'

# ---------------------------------------------------------------------
# Settings for GPT

settings['gun_field:value'] = 1.0   # asymptotic field for large r
settings['gun_field:units'] = 'MV/m'

settings['plummer_radius:value'] = 1.0e-3
settings['plummer_radius:units'] = 'nm'

settings['cathode_z_offset:value'] = 3
settings['cathode_z_offset:units'] = 'nm'

settings['tmax'] = 1e-9
settings['zmax'] = 1.1e-6 

# ---------------------------------------------------------------------
# RNG settings
rng = np.random.default_rng()

In [35]:
# ---------------------------------------------------------------------
# Make initial distribution

n_particles = 2000

verbose=True

# ---------------------------------------------------------------------

settings_copy = copy.copy(settings)
settings_copy['n_particle'] = n_particles
# ---------------------------------------------------------------------

if (settings['cathode_type'] == 'metal'):
    PG_initial = MakeMetalParticleGroup(settings_copy, DISTGEN_INPUT_FILE=DISTGEN_INPUT_FILE, verbose=verbose, only_survivors=settings_copy['only_survivors'], rng=rng)
elif (settings['cathode_type'] == 'semiconductor'):
    PG_initial = MakeSemiconductorParticleGroup(settings_copy, DISTGEN_INPUT_FILE=DISTGEN_INPUT_FILE, verbose=verbose, only_survivors=settings_copy['only_survivors'], rng=rng)
elif (settings['cathode_type'] == 'distgen'):
    if (not settings_copy['only_survivors']):
        raise ValueError("ERROR: only_survivors must be used with distgen")
    PG_initial = MakeEnergyOffsetParticleGroup(settings_copy, DISTGEN_INPUT_FILE=DISTGEN_INPUT_FILE, verbose=verbose)
else:
    raise ValueError("ERROR: bad cathode type")

PG_initial.weight = ELEMENTARY_CHARGE # Just to make sure, not really needed

PG_run = copy.deepcopy(PG_initial)
if (settings['cathode_wall'] == 1):
    PG_run.z = -settings_copy['H_cylinder']

Adding settings["gun_field"] = 1000000.0 for use in GPT
Adding settings["cathode_z_offset"] = 3.0000000000000004e-09 for use in GPT
Adding settings["plummer_radius"] = 1.0000000000000002e-12 for use in GPT
Peak potential barrier at z = 16 nm
Eexc at surface = 0.2199970441667078, Eexc at peak = 0.134946864817795


In [66]:
if (settings['cathode_wall'] == 1):
    print("Creating BEM Well geometry...")
    geometry = CylindricalWellBEMGeometry(E_gun=-settings_copy["gun_field"], z0=settings_copy["cathode_z_offset"], R=settings_copy['R_cylinder'], H=settings_copy['H_cylinder'],
                                      field_max_length=2.0e-9,
                                      image_max_length=2.0e-9,
                                      rim_fillet_radius=5.0e-9,
                                      bottom_fillet_radius=1.5e-9
                                     )
else:
    print("Using analytic flat cathode geometry...")
    geometry = FlatCathode(E_gun=-settings_copy["gun_field"], z0=settings_copy["cathode_z_offset"]) 

Creating BEM Well geometry...


In [ ]:
R = settings_copy['R_cylinder']
H = settings_copy['H_cylinder']

def add_minus_x(p):
    pm = copy.copy(p)
    pm[:,0] = -pm[:,0]
    p = np.append(p[::-1], pm, axis=0)
    return p

%matplotlib inline
fig, ax = plt.subplots(figsize=(10, 6))
real_profile = add_minus_x(geometry.real_profile*1e9)
image_profile = add_minus_x(geometry.image_solution.profile*1e9) if geometry.image_solution is not None else None
plot_profiles(ax, real_profile, image_profile)
ax.set_xlim(-3*R*1e9, 3*R*1e9)
ax.set_ylim(-1.8*H*1e9, 0.1*H*1e9)
ax.set_xlabel("x (nm)")
ax.set_ylabel("z (nm)")
ax.set_aspect("equal") 
plt.show()
%matplotlib widget

In [ ]:
R = settings_copy['R_cylinder']
H = settings_copy['H_cylinder']
R_grid, Z_grid, V_grid = static_potential_grid(geometry.field_solver, r_max=3 * R, z_range=(-1.5*H, 2 * H))

source_pos = [0.8*R, -0.5*H]
if geometry.image_solution is not None:
    R_grid_imag, Z_grid_imag, V_grid_imag = image_potential_grid(
            geometry.image_solution,
            source_r0=source_pos[0], source_z0=source_pos[1],  # pick a point near where particles are actually emitted
            r_max=3 * R, z_range=(-1.5*H, 2 * H),
            charge=-ELEMENTARY_CHARGE, query_phi=0.0,
        )

r_bound = geometry.real_profile[:,0]
z_bound = geometry.real_profile[:,1]

In [ ]:
%matplotlib inline
fig, ax = plt.subplots()

cf = ax.contourf(
    R_grid*1e9, Z_grid*1e9, 1e3*V_grid,
    levels=20, cmap="RdBu_r"
)

cbar = fig.colorbar(
    cf,
    ax=ax,
    label="Voltage (mV)",
    shrink=0.75,   # height relative to plot
    aspect=30,     # larger = thinner
    pad=0.05       # gap between plot and colorbar
)

ax.plot(r_bound*1e9, z_bound*1e9, 'k-')

ax.set_xlabel("r (nm)")
ax.set_ylabel("z (nm)")
ax.set_aspect("equal")

ax.set_xlim(0, 3*R*1e9)
plt.show()
%matplotlib widget

In [ ]:
%matplotlib inline
if geometry.image_solution is not None:
    fig, ax = plt.subplots(figsize=(6, 6))
    cf = ax.contourf(R_grid_imag*1e9, Z_grid_imag*1e9, V_grid_imag*1e3, levels=20, cmap="RdBu_r")
    
    cbar = fig.colorbar(
        cf,
        ax=ax,
        label="Voltage (mV)",
        shrink=0.75,   # height relative to plot
        aspect=30,     # larger = thinner
        pad=0.05       # gap between plot and colorbar
    )
    
    ax.plot(source_pos[0]*1e9, source_pos[1]*1e9, marker="o", color="k", ms=7, ls="none")
    
    ax.plot(r_bound*1e9, z_bound*1e9, 'k-')

    ax.set_xlim(0, 3*R*1e9)
    ax.set_xlabel("r (nm)"); ax.set_ylabel("z (nm)"); ax.set_aspect("equal")
    plt.show()
else:
    print("geometry was built with z0=None -- no image-charge solution to plot")
%matplotlib widget

## Run

In [50]:
t_max_plot = 1.0e-12
t_out = np.linspace(0, t_max_plot, 1000)
#t_out = None

z_screen = 1e-6

n_emit = 1

tracer = SpecificParticleTracer(
    initial_particles=PG_run,
    n_emit=n_emit,
    geometry=geometry,
    screens=[z_screen],   # 1 micron above the tip
    t_out=t_out,
    z_max=settings_copy['zmax'],   # stop tracking a particle once it's well past the screen
    t_max=settings_copy['tmax'],
    plummer_radius=settings_copy["plummer_radius"],
    #backend='gpu',
    n_workers=max_workers,
    rtol=1.0e-7,
    atol=1.0e-12,
)

t_run_gpt = time.time()
screens, trajectories = tracer.run(verbose=True)
print(f'Elapsed: {time.time() - t_run_gpt} s')
print(f"{len(screens[-1])} of {len(PG_run)} particles reached the screen")

by_id = collect_trajectories(trajectories, PG_run)
print(f"tracked {len(by_id)} particle trajectories")

99 groups split across 90 worker processes (backend=cpu)
Elapsed: 87.0894558429718 s
77 of 99 particles reached the screen
tracked 99 particle trajectories


In [ ]:
R = settings_copy['R_cylinder']
H = settings_copy['H_cylinder']

%matplotlib inline
fig, ax = plt.subplots(figsize=(6, 6))

z_plot_max = H * 1
r_plot_max = R * 3

if hasattr(geometry, 'real_profile'):
    r_bound = geometry.real_profile[:,0]
    z_bound = geometry.real_profile[:,1]
    ax.plot(r_bound * 1e9, z_bound * 1e9, color="0.6", lw=2)

for pid, (t, pos, mom) in by_id.items():
    ax.plot(np.sqrt(pos[:, 0]**2 + pos[:, 1]**2) * 1e9, pos[:, 2] * 1e9, lw=0.8, alpha=0.7)

ax.set_xlabel("r [nm]")
ax.set_ylabel("z [nm]")
ax.set_xlim(0, r_plot_max*1e9)
ax.set_ylim(-H*1e9, z_plot_max*1e9)
ax.set_aspect("equal")
plt.show()
%matplotlib widget

In [ ]:
r_bound = geometry.real_profile[:, 0]
z_bound = geometry.real_profile[:, 1]

R = settings_copy['R_cylinder']
H = settings_copy['H_cylinder']

%matplotlib inline

from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(6, 6))

z_plot_max = H * 1
r_plot_max = R * 3

ax.plot(r_bound * 1e9, z_bound * 1e9, color="0.6", lw=2)

# One color for each emission order within a group
colors = plt.cm.plasma(np.linspace(0, 1, n_emit))

# Sort by particle ID first, so consecutive IDs form an emission group
particles = sorted(by_id.items(), key=lambda x: x[0])

for i in range(0, len(particles), n_emit):
    group = particles[i:i + n_emit]

    # Order particles within this group by their first recorded time
    group = sorted(group, key=lambda x: x[1][0][0])

    for j, (pid, (t, pos, mom)) in enumerate(group):
        ax.plot(
            np.sqrt(pos[:, 0]**2 + pos[:, 1]**2) * 1e9,
            pos[:, 2] * 1e9,
            color=colors[j],
            lw=0.8,
            alpha=0.7,
        )

ax.set_xlabel("r [nm]")
ax.set_ylabel("z [nm]")
ax.set_xlim(0, r_plot_max * 1e9)
ax.set_ylim(-H * 1e9, z_plot_max * 1e9)
ax.set_aspect("equal")

order_labels = ["First", "Second", "Third", "Fourth", "Fifth", "Sixth", "Seventh"]

legend_handles = [
    Line2D([0], [0], color=colors[i], lw=2, label=order_labels[i])
    for i in range(n_emit)
]

ax.legend(handles=legend_handles, title="Emission order")

plt.show()
%matplotlib widget

## Multi-run

In [37]:
# ---------------------------------------------------------------------
# Initialize arrays for new run

min_particles = 0
max_particles = 20

# ---------------------------------------------------------------------

M = np.zeros((max_particles+1,max_particles+1))
counts_n = np.zeros((max_particles+1,max_particles+1))
n_runs_n = np.zeros(max_particles+1, dtype=int)
gpt_data_n = [None]*(max_particles+1)

M[0,0] = 1.0
counts_n[0,0] = 1

if (max_particles == min_particles):
    values_to_scan_all = np.array([max_particles])
else:
    values_to_scan_all = np.append(1, np.arange(np.max([2,min_particles]), max_particles+1))

values_to_scan_all = np.array([int(n) for n in values_to_scan_all if int(n) != 0])

attempts = {int(num_par): 0 for num_par in values_to_scan_all}

In [ ]:
# ---------------------------------------------------------------------
# Main run loop
#
# Builds the matrix of N_injected vs N_emitted
# gpt_data_n : list of gpt_data objects for n injected electrons 
# ids from 1 to n_particle are in the first bunch, etc.

rel_accuracy = 0.1

n_runs_start = 90 * 6
n_runs_min_add = 90
n_runs_max_times_more = 10
n_runs_max_total = 1000000

max_attempts = 10
sleep_after_crash = 2

# ---------------------------------------------------------------------

t_run_gpt = time.time()

while len(get_values_to_scan()) > 0:

    values_to_scan = get_values_to_scan()

    try:

        for num_par in values_to_scan:

            print(f'Beginning {num_par} particles: --------------------------')

            while not is_done(num_par):

                n_runs_this_batch = get_next_n_runs(counts_n[num_par,:])

                if (n_runs_this_batch == 0):
                    break

                print(f'Running {n_runs_this_batch} trials')

                settings_run = copy.copy(settings)
                settings_run['n_particle'] = int(n_runs_this_batch * num_par)

                PG_initial = make_initial_particle_group(settings_run, rng=rng)

                t_out = None
                z_screen = 1e-6
                
                tracer = SpecificParticleTracer(
                    initial_particles=PG_initial,
                    n_emit=int(num_par),
                    geometry=geometry,
                    screens=[z_screen],
                    t_out=t_out,
                    z_max=settings_copy['zmax'],   
                    t_max=settings_copy['tmax'],
                    plummer_radius=settings_copy["plummer_radius"],
                    #backend='gpu',
                    n_workers=max_workers,
                    rtol=1.0e-7,
                    atol=1.0e-12
                )
                
                screens, trajectories = tracer.run(verbose=True)

                gpt_data = GPT(gpt_bin='.')
                gpt_data.output["particles"] = [copy.deepcopy(PG_initial), copy.deepcopy(screens[0])]
                gpt_data.output["n_tout"] = 0
                gpt_data.output["n_screen"] = 2
                gpt_data.input = {"lines": [],"variables": {}}

                settings_run['zmax'] = z_screen # Fixes problem in next function
                counts_this_batch = get_counts_from_gpt_data(gpt_data, num_par, n_runs_this_batch, settings_run)

                id_offset = n_runs_n[num_par] * num_par
                gpt_data_n[num_par] = append_gpt_data(gpt_data_n[num_par], gpt_data, id_offset, settings_run)

                counts_n[num_par,:] = counts_n[num_par,:] + counts_this_batch
                n_runs_n[num_par] = n_runs_n[num_par] + n_runs_this_batch
                M[num_par,:] = counts_n[num_par,:] / np.sum(counts_n[num_par,:])

                pmax = np.max(M[num_par,1:])
                rel_err = get_row_rel_err(counts_n[num_par,:])

                max_n_emit = 0
                if (num_par > 2):
                    max_n_emit = get_fano_curve(M[0:num_par,0:num_par], np.array([get_avg_e_max_from_M(M[0:num_par,0:num_par])]))[0][0]
                print(f'n_runs = {n_runs_n[num_par]}, pmax = {pmax:.5g}, rel_err = {rel_err:.5g}, max_n_emit = {max_n_emit:.5g}')

                attempts[num_par] = 0

    except Exception as err:

        attempts[num_par] += 1

        print(f'GPT failed for {num_par} particles')
        print(f'Attempt {attempts[num_par]} of {max_attempts}')
        print(err)

        if (attempts[num_par] >= max_attempts):
            raise RuntimeError(f'Giving up on {num_par} particles after {max_attempts} attempts') from err

        time.sleep(sleep_after_crash)

print(f'Elapsed: {time.time() - t_run_gpt} s')
print(M)
print(n_runs_n)

Beginning 32 particles: --------------------------
Running 540 trials
540 groups split across 90 worker processes (backend=cpu)
n_runs = 540, pmax = 0.37407, rel_err = 0.055665, max_n_emit = 1.1196
Beginning 33 particles: --------------------------
Running 540 trials
540 groups split across 90 worker processes (backend=cpu)
n_runs = 540, pmax = 0.4037, rel_err = 0.0523, max_n_emit = 1.1462
Beginning 34 particles: --------------------------
Running 540 trials


In [ ]:
# ---------------------------------------------------------------------
# Save output
# Saves the matrix M, counts_n, n_runs_n, and the set of gpt_data objects

base_dir = '/nfs/bbl/online/coulomb/saves'
save_gpt_n = True

extra_prefix = 'SPT_meff_'  # meff_

# ---------------------------------------------------------------------

z0 = getValueFromSettings(settings, 'cathode_z_offset', 'nm', modify_settings=False, verbose=False)
d = 4 * getValueFromSettings(settings, 'r_dist:sigma_xy', 'nm', modify_settings=False, verbose=False)
st = getValueFromSettings(settings, 't_dist:sigma_t', 'fs', modify_settings=False, verbose=False)
Ef = getValueFromSettings(settings, 'gun_field', 'MV/m', modify_settings=False, verbose=False)
Egamma = getValueFromSettings(settings, 'photon_energy', 'eV', modify_settings=False, verbose=False)

if (settings['cathode_wall'] == 1):
    H = settings['H_cylinder']*1e9
else:
    H = 0

dir_prefix = 'full'
if (settings['only_survivors']):
    dir_prefix = 'surv'

cath_type = settings['cathode_type']
if (cath_type == 'semiconductor'):
    Eother = getValueFromSettings(settings, 'electron_affinity', 'eV', modify_settings=False, verbose=False) 
elif (cath_type == 'metal'):
    Eother = getValueFromSettings(settings, 'work_function', 'eV', modify_settings=False, verbose=False)
elif (cath_type == 'distgen'):
    Egamma = getValueFromSettings(settings, 'start:MTE', 'eV', modify_settings=False, verbose=False)
    Eother = 0
else:
    raise ValueError("ERROR: bad cathode type")

run_label = f'adapt_ra={rel_accuracy}'

file_dir = os.path.join(
    base_dir,
    f"{extra_prefix}{settings['cathode_type'][0]}_{dir_prefix}_{run_label}_D={d}_H={H}_Ef={Ef}_sigt={st}_z0={z0:.2f}_E={Egamma}_{Eother}"
)

try:
    os.mkdir(file_dir)
except OSError as error:
    print('Warning: Directory already exists')

print(f"Saving to: {os.path.join(file_dir, 'M.txt')}")
np.savetxt(os.path.join(file_dir, 'M.txt'), M)

if (save_gpt_n):

    for num_par,g in enumerate(gpt_data_n):

        if (g is None):
            if (num_par != 0):
                print(f'Warning: gpt_data_n[{num_par}] is None')
            continue

        file_to_save = os.path.join(file_dir, f'gpt_data_{num_par}.h5')
        print(f'Saving to: {file_to_save}')
        g.archive(h5=file_to_save)

with open(os.path.join(file_dir, 'settings.json'), "w") as f:

    settings_save = copy.copy(settings)

    settings_save['min_particles'] = min_particles
    settings_save['max_particles'] = max_particles

    settings_save['rel_accuracy'] = rel_accuracy
    settings_save['n_runs_start'] = n_runs_start
    settings_save['n_runs_min_add'] = n_runs_min_add
    settings_save['n_runs_max_times_more'] = n_runs_max_times_more

    settings_save['save_gpt_n'] = save_gpt_n
    settings_save['n_runs_n'] = n_runs_n.astype(int)
    settings_save['counts_n'] = counts_n.astype(int)

    json.dump(settings_save, f, indent=2, default=lambda x: x.tolist())

In [ ]:
%matplotlib inline
fig, ax = plt.subplots()
plot_fano_with_error(counts_n, avg_e_max=None, use_electrons_escaping=True,
                         fig_ax=(fig,ax), color='r', label='Data',
                         n_trials=50, ci=0.95, rng=None)
plt.show()
%matplotlib widget

In [ ]:
%matplotlib inline
fig, ax = plt.subplots()
plot_merit(counts_n, avg_e_max=None, fig_ax=(fig,ax), color='r', label='Data',
               show_poisson=True, xlim=[0.001,1], n_trials=50, ci=0.95,
               rng=None)
plt.show()
%matplotlib widget

## Continue Old Run

In [ ]:
save_dir = '/nfs/bbl/online/coulomb/saves/'
print(f'{len(os.listdir(save_dir))} subdirectory:')
os.listdir(save_dir)

In [64]:
(counts_n, settings, gpt_data_n) = load_data('SPT_meff_s_full_adapt_ra=0.02_D=9.999999999999998_H=10.0_Ef=0.1_sigt=10_z0=3.00_E=1.5_-0.0', verbose=False)
M = M_from_C(counts_n)
n_runs_n = counts_from_C(counts_n)

print(f'len(M) = {len(M)}')

if ('only_survivors' not in settings):
    settings['only_survivors'] = False
    print(f"Setting 'only_survivors' to {settings['only_survivors']}")

Missing GPT data for: [29, 30]
len(M) = 31


/tmp/ipykernel_939009/2437595535.py:28: RuntimeWarning: invalid value encountered in divide
  return C / np.sum(C, axis=1, keepdims=True)


In [65]:
# Initialize things for a restart, and check consistency

min_particles = 0
max_particles_new = 40

z_screen = 1.0e-6

old_max_particles = M.shape[0] - 1

#if (max_particles_new < old_max_particles):
#    raise ValueError('max_particles_new is smaller than the loaded matrix')

if (max_particles_new > old_max_particles):
    
    M_old = M
    counts_n_old = counts_n
    n_runs_n_old = n_runs_n
    gpt_data_n_old = gpt_data_n

    M = np.zeros((max_particles_new+1, max_particles_new+1))
    counts_n = np.zeros((max_particles_new+1, max_particles_new+1))
    n_runs_n = np.zeros(max_particles_new+1, dtype=int)

    M[:old_max_particles+1, :old_max_particles+1] = M_old
    counts_n[:old_max_particles+1, :old_max_particles+1] = counts_n_old
    n_runs_n[:old_max_particles+1] = n_runs_n_old

    gpt_data_n = gpt_data_n_old + [None]*(max_particles_new - old_max_particles)

max_particles = max_particles_new

M[0,0] = 1.0
counts_n[0,0] = 1

if (max_particles == min_particles):
    values_to_scan_all = np.array([max_particles])
else:
    values_to_scan_all = np.append(1, np.arange(np.max([2,min_particles]), max_particles+1))

values_to_scan_all = np.array([int(n) for n in values_to_scan_all if int(n) != 0])

attempts = {int(num_par): 0 for num_par in values_to_scan_all}

settings_run = copy.copy(settings)
settings_run['zmax'] = z_screen # Fixes problem in next function

bad_rows, repaired_rows = check_loaded_rows(
    M,
    counts_n,
    n_runs_n,
    gpt_data_n,
    settings_run,
    check_gpt=True,
    reset_bad=True
)

All loaded rows look self-consistent
